<a href="https://colab.research.google.com/github/mehmoodanas/Bootcamp/blob/main/Day-7/Transformers_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 7 · Transformer assignment

**Core work: 30–40 minutes, after setup.** Work alone or in pairs. This notebook runs independently of the demonstration; it does not rely on variables from another kernel. Use the [guided lab](../Transformers_what_can_they_do_.ipynb) as a reference.

**Submit:** this notebook with your completed code, your own outputs and short written answers. Required evidence is a task/model/architecture table, token-count comparison, sentiment stress test, two bounded sampled continuations, a greedy comparison and one limitation. Exact generated text need not match another learner's.

Cells marked `TODO` are deliberately incomplete. Running the starter notebook prints reminders rather than pretending the exercises are solved. Short hints follow each task. Instructor solutions are in a separate file and are not needed to begin.

## Setup - do this before the session

Use a **fresh Python 3.12 environment** with **PyTorch 2.6.0 CPU** and **Transformers 4.57.1**. These are the course pins, not a claim that they are the newest releases. Do not upgrade to v5 during the session: some older pipeline tasks, including summarization and translation, were removed from that API. A v5 migration needs different task code and a new smoke test. [Versioned pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines) · [Current translation model instructions](https://huggingface.co/Helsinki-NLP/opus-mt-fr-en)

The first model load downloads public weights and tokenizer files. The three core weight files total about **0.89 GB**, excluding packages and cache overhead. Allow **at least 3 GB of free disk space** for the core environment/cache. A recent laptop with **4 CPU cores and 8 GB RAM** is the planning assumption. Budget **5–20 minutes for first downloads and setup**, and roughly **1–5 minutes of computation after caching**; these are planning estimates, not benchmarks or guarantees. Network speed, package installation and hardware can change them substantially.

Examples use invented, non-sensitive text. The code runs model inference locally; downloading weights requires a Hub connection. Do not paste client material into a public Hub widget. No Hugging Face login or paid inference API is required for these public checkpoints.

In [ ]:
# Optional one-time installation in a fresh notebook kernel, then restart it:
# %pip install "torch==2.6.0" --index-url https://download.pytorch.org/whl/cpu
# %pip install "transformers==4.57.1"
# On macOS, use this torch install line instead of the CPU-index line:
# %pip install "torch==2.6.0"

In [3]:
import os
import platform
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version

OFFLINE = False  # Change before imports; restart the kernel after changing it.
if OFFLINE:
    os.environ['HF_HUB_OFFLINE'] = '1'
os.environ.setdefault('HF_HOME', str(Path.home() / '.cache' / 'huggingface'))
LOCAL_FILES_ONLY = OFFLINE or os.environ.get('HF_HUB_OFFLINE') == '1'

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
import torch
from transformers import AutoTokenizer, pipeline, set_seed

torch.set_num_threads(min(4, os.cpu_count() or 1))
SEED = 42
set_seed(SEED)
print('Python:', platform.python_version())
print('Device: CPU; threads:', torch.get_num_threads())
print('HF_HOME:', os.environ['HF_HOME'])
print('Local cache only:', LOCAL_FILES_ONLY)

torch: 2.6.0+cpu
transformers: 4.57.1
Python: 3.13.18
Device: CPU; threads: 2
HF_HOME: /root/.cache/huggingface
Local cache only: False


In [4]:
from importlib.metadata import PackageNotFoundError, version

def require_versions(required):
    for package, expected in required.items():
        try:
            actual = version(package)
        except PackageNotFoundError as exc:
            raise RuntimeError(f'Install {package}=={expected} in this kernel, then restart it.') from exc
        print(f'{package}: {actual}')
        if actual.split('+')[0] != expected:
            raise RuntimeError(f'This lab pins {package}=={expected}; use the course environment and restart.')

try:
    require_versions({'torch': '2.6.0', 'transformers': '4.57.1'})
    print('Environment checks passed successfully!')
except Exception as e:
    print('Environment Check Failed:', str(e))

torch: 2.6.0+cpu
transformers: 4.57.1
Environment checks passed successfully!


In [5]:
MODEL_SPECS = {'sentiment': {'task': 'sentiment-analysis',
               'model': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
               'revision': '714eb0fa89d2f80546fda750413ed43d93601a13'},
 'mask': {'task': 'fill-mask',
          'model': 'distilbert/distilbert-base-uncased',
          'revision': '12040accade4e8a0f71eabdb258fecc2e7e948be'},
 'generation': {'task': 'text-generation',
                'model': 'distilbert/distilgpt2',
                'revision': '2290a62682d06624634c1f46a6ad5be0f47f38aa'}}

def load_pipeline(name, **task_options):
    spec = MODEL_SPECS[name]
    tokenizer = AutoTokenizer.from_pretrained(
        spec['model'], revision=spec['revision'],
        local_files_only=LOCAL_FILES_ONLY, trust_remote_code=False,
    )
    result = pipeline(
        task=spec['task'], model=spec['model'], tokenizer=tokenizer,
        revision=spec['revision'], framework='pt', device=-1,
        trust_remote_code=False,
        model_kwargs={'use_safetensors': True, 'local_files_only': LOCAL_FILES_ONLY},
        **task_options,
    )
    print(f"{name}: {spec['model']} @ {spec['revision'][:7]}")
    print(type(result.model).__name__, '/', type(result.tokenizer).__name__)
    return result

## Task 1 · Match the parts (5 minutes)

Complete the table. Use the three checkpoints in `MODEL_SPECS`. In the last column, describe the result's meaning rather than just its Python type.

| Task | Checkpoint | Architecture family | What the output means |
|---|---|---|---|
| Sentiment | distilbert/distilbert-base-uncased-finetuned-sst-2-english | Encoder | It reacts acoording to the word/sentence it is positive or negative and gives a score |
| Fill-mask | distilbert/distilbert-base-uncased | Encoder | It gueasses rge hidden word and also gives score |
| Text generation | distilbert/distilgot2 | Decoder | It decodes the sentence and write new words after our sentence |

**Explain in 2–3 sentences:** why does selecting a base BERT-like encoder not automatically give us a trained sentiment classifier? What must match between a tokenizer and a checkpoint?

**Success:** all three mappings are complete; your explanation distinguishes architecture, learned weights, output head and tokenizer. **Hint:** read the model card's task and training description; `pipeline` is a wrapper, not an architecture.

**Your answer:** A base BERT model basically only learned to guess hidden words, so it does not know anything about positive or negative feelings yet. The architecture is just the shape of the model, and what it can really do comes from the trained weights and the output head. So to get a sentiment classifier, the model must be fine-tuned on labeled sentiment data, like the SST-2 movie reviews used for my sentiment model. The tokenizer and the model must actually match, because they need the same vocabulary, so every token ID means the same word piece the model learned in training.

## Task 2 · Make tokens visible (7 minutes)

Load the mask pipeline in the next cell. For `token_text`, compute a whitespace word count, token IDs **without special tokens**, and a token count. Print the token strings too. Then construct one sentence using `unmasker.tokenizer.mask_token` and request its top three completions.

Change one context word and run the mask prediction again. Record the candidate token and score that changed most noticeably. Do not guess a prediction before running it.

**Success:** you show both counts and the actual IDs/token strings; two inputs each contain exactly one valid mask; each result has three candidates. Explain why top-three scores need not sum to one.

**Hint:** `tokenizer(text, add_special_tokens=False)` returns a dictionary containing `input_ids`; `convert_ids_to_tokens` makes those IDs readable. The tokenizer exposes `mask_token`.

In [9]:
unmasked = load_pipeline('mask')
token_text = "Re-tokenization isn't always one word per token."
word_count = len(token_text.split())     # TODO
token_ids = unmasked.tokenizer(token_text, add_special_tokens=False)['input_ids']       # TODO
token_count = len(token_ids)     # TODO
masked_text = f"I am going to {unmasked.tokenizer.mask_token} to do exercise and then i will go to {unmasked.tokenizer.mask_token} to buy something and do shopping and then i will {unmasked.tokenizer.mask_token} movie in the cinema"     # TODO: use the tokenizer's mask marker
mask_results = unmasked(masked_text, top_k=3)    # TODO: call unmasker for three candidates

if any(value is None for value in [word_count, token_ids, token_count, masked_text, mask_results]):
    print('Complete Task 2 TODOs, then rerun this cell.')
else:
    print('Words:', word_count, 'Tokens:', token_count)
    print('Token IDs:', token_ids)
    print('Token strings:', unmasked.tokenizer.convert_ids_to_tokens(token_ids))
    print(mask_results)
    assert masked_text.count(unmasked.tokenizer.mask_token) == 3
    assert len(mask_results) == 3
    assert token_count == len(token_ids)

Device set to use cpu


mask: distilbert/distilbert-base-uncased @ 12040ac
DistilBertForMaskedLM / DistilBertTokenizerFast
Words: 7 Tokens: 13
Token IDs: [2128, 1011, 19204, 3989, 3475, 1005, 1056, 2467, 2028, 2773, 2566, 19204, 1012]
Token strings: ['re', '-', 'token', '##ization', 'isn', "'", 't', 'always', 'one', 'word', 'per', 'token', '.']
[[{'score': 0.32971489429473877, 'token': 2175, 'token_str': 'go', 'sequence': '[CLS] i am going to go to do exercise and then i will go to [MASK] to buy something and do shopping and then i will [MASK] movie in the cinema [SEP]'}, {'score': 0.04372011497616768, 'token': 4553, 'token_str': 'learn', 'sequence': '[CLS] i am going to learn to do exercise and then i will go to [MASK] to buy something and do shopping and then i will [MASK] movie in the cinema [SEP]'}], [{'score': 0.04259141907095909, 'token': 8955, 'token_str': 'mumbai', 'sequence': '[CLS] i am going to [MASK] to do exercise and then i will go to mumbai to buy something and do shopping and then i will [MASK

In [10]:
changed_masked_text = f"Today my day is good. I will try to [MASK] hard to get success in the bootcamp and get an internship"  # TODO: change one context word
changed_mask_results = unmasked(changed_masked_text, top_k=3) # TODO: rerun the same model with top_k=3
if changed_mask_results is None:
    print('Complete the second input and compare its observed ranking.')
else:
    print(changed_masked_text, changed_mask_results)

Today my day is good. I will try to [MASK] hard to get success in the bootcamp and get an internship [{'score': 0.742336630821228, 'token': 2147, 'token_str': 'work', 'sequence': 'today my day is good. i will try to work hard to get success in the bootcamp and get an internship'}, {'score': 0.13672859966754913, 'token': 3046, 'token_str': 'try', 'sequence': 'today my day is good. i will try to try hard to get success in the bootcamp and get an internship'}, {'score': 0.024827899411320686, 'token': 2954, 'token_str': 'fight', 'sequence': 'today my day is good. i will try to fight hard to get success in the bootcamp and get an internship'}]


**Your interpretation:** TODO - include the two counts, one changed candidate and an explanation of the scores.

## Task 3 · Stress-test a classifier (7 minutes)

Before running the model, label the four examples yourself as positive, negative, neutral or ambiguous. Then call the sentiment pipeline on the entire list, with input truncation enabled and a 128-token input cap. Compare your assessment with the model's available labels.

**Success:** four observed predictions with scores; one disagreement or ambiguous case discussed; one sentence explaining why a high score does not validate use on client tickets. If you agree with all four outputs, add an invented neutral or mixed-sentiment example and discuss the forced binary choice.

**Hint:** this checkpoint has no neutral output class. Inspect `sentiment.model.config.id2label`. A tiny hand-picked sample illustrates a failure mode; it is not a benchmark.

In [11]:
sentiment = load_pipeline('sentiment')
examples = [
    'The explanation was clear and useful.',
    'The application crashes every time I open it.',
    "The meeting starts at two o'clock.",
    'I loved the tutor, but the exercises were frustrating.',
]
my_labels = ['Positive', 'Negative', 'Nuetral', 'Ambiguous']              # TODO: four judgments before model inference
student_predictions = sentiment(examples) # TODO: one pipeline call on examples
print('Available model labels:', sentiment.model.config.id2label)
if student_predictions is None or len(my_labels) != len(examples):
    print('Complete your judgments and the pipeline call.')
else:
    assert len(student_predictions) == len(examples)
    for text, human, model_result in zip(examples, my_labels, student_predictions):
        print({'text': text, 'my_judgment': human, 'model': model_result})

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Device set to use cpu


sentiment: distilbert/distilbert-base-uncased-finetuned-sst-2-english @ 714eb0f
DistilBertForSequenceClassification / DistilBertTokenizerFast
Available model labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
{'text': 'The explanation was clear and useful.', 'my_judgment': 'Positive', 'model': {'label': 'POSITIVE', 'score': 0.9997910857200623}}
{'text': 'The application crashes every time I open it.', 'my_judgment': 'Negative', 'model': {'label': 'NEGATIVE', 'score': 0.9997283816337585}}
{'text': "The meeting starts at two o'clock.", 'my_judgment': 'Nuetral', 'model': {'label': 'POSITIVE', 'score': 0.9768263697624207}}
{'text': 'I loved the tutor, but the exercises were frustrating.', 'my_judgment': 'Ambiguous', 'model': {'label': 'NEGATIVE', 'score': 0.9912057518959045}}


**Your interpretation:** TODO - identify an ambiguous or failed case, and propose one concrete check before applying this checkpoint to a different domain.

## Task 4 · Bound generation and compare decoding (10 minutes)

Load the generation model and run the provided reporting helper. Write a call that generates **two sampled continuations with at most 15 new tokens each**. Use a seed of 42, one beam and deliberate sampling parameters. Request token IDs with `return_tensors=True`; pass the tokenizer's EOS ID as `pad_token_id` for this model.

Then generate **one greedy continuation** with the same prompt and token cap. Print all three continuations with both token counts and whitespace word counts.

**Success:** two sampled results, one greedy result, and each count at most 15. Your explanation says “maximum new tokens,” allows early stopping and does not require exactly 15 words, two complete sentences or different sampled texts.

**Hint:** inspect `max_new_tokens`, `num_return_sequences` and `do_sample`. Greedy decoding with one beam uses one return sequence. Tune `temperature` and `top_p` for sampling. For the greedy call, use neutral `temperature=1.0` to reset this v4 pipeline's inherited 0.7 default and avoid an ignored-parameter warning; it has no sampling effect when `do_sample=False`.

In [12]:
def describe_generations(results, prompt, tokenizer):
    """Count original generated IDs, rather than re-tokenizing decoded text."""
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    rows = []
    for result in results:
        all_ids = result['generated_token_ids']
        assert all_ids[:len(prompt_ids)] == prompt_ids
        added_ids = all_ids[len(prompt_ids):]
        # DistilGPT2 uses EOS as padding here; ignore padding after the first EOS.
        if tokenizer.eos_token_id in added_ids:
            added_ids = added_ids[:added_ids.index(tokenizer.eos_token_id) + 1]
        continuation = tokenizer.decode(added_ids, skip_special_tokens=True)
        rows.append({
            'continuation': continuation,
            'new_tokens_including_eos': len(added_ids),
            'whitespace_words': len(continuation.split()),
        })
    return rows

In [13]:
generator = load_pipeline('generation')   # load DistilGPT2
prompt = 'To prepare a useful training workshop, we should'
set_seed(SEED)   # seed 42, so the random sampling can be repeated

sampled_results = generator(prompt, max_new_tokens=15, num_return_sequences=2,
                            do_sample=True, num_beams=1, temperature=0.8, top_p=0.9,
                            return_tensors=True, pad_token_id=generator.tokenizer.eos_token_id)   # two sampled continuations

greedy_results = generator(prompt, max_new_tokens=15, num_return_sequences=1,
                           do_sample=False, num_beams=1, temperature=1.0, return_tensors=True,
                           pad_token_id=generator.tokenizer.eos_token_id)   # one greedy continuation

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cpu


generation: distilbert/distilgpt2 @ 2290a62
GPT2LMHeadModel / GPT2TokenizerFast


In [14]:
if sampled_results is None or greedy_results is None:
    print('Complete both generation calls, then rerun this check.')
else:
    sampled_rows = describe_generations(sampled_results, prompt, generator.tokenizer)
    greedy_rows = describe_generations(greedy_results, prompt, generator.tokenizer)
    assert len(sampled_rows) == 2
    assert len(greedy_rows) == 1
    assert all(1 <= row['new_tokens_including_eos'] <= 15 for row in sampled_rows + greedy_rows)
    print('Sampled:', sampled_rows)
    print('Greedy:', greedy_rows)

Sampled: [{'continuation': ' try to introduce some basic knowledge about the different methods used for training.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}, {'continuation': ' be able to use this tool to help us train in a more efficient manner', 'new_tokens_including_eos': 15, 'whitespace_words': 15}]
Greedy: [{'continuation': ' be able to use the following techniques to prepare a useful training workshop.\n', 'new_tokens_including_eos': 15, 'whitespace_words': 13}]


**Your interpretation:** TODO - compare the observed outputs and explain (a) tokens versus words, (b) sampling versus greedy, and (c) why neither method guarantees factuality. What would an application need to do if it genuinely required exactly 15 whitespace-delimited words?

(a) As we can see in the results. there were 7 words and 15 tokens so it means one word can be with 2 3 or more tokens and one word is not equal to one token.

(b) Greedy always picks the most likely token, so it basically repeated words from my prompt. Sampling picks randomly from likely tokens, so my two sampled texts were different.

(c) Both only predict the next likely word and do not check if it is true. So the text can sound good but still be wrong.

## Task 5 · Make a model decision (5 minutes)

A team wants to use one of these checkpoints on a new task. Choose **one** scenario and write 4–6 sentences:

- Route short English customer-support messages to three departments.
- Suggest a continuation for public training copy.
- Extract named organizations from public event announcements.

Name a suitable task and model family, identify a mismatch with at least one of the core checkpoints, give one test case, and state one evaluation measure or review criterion. Also identify a case the system should hand to a human. You do not need to download another model.

**Success:** your recommendation follows the task and evidence; it does not use a sentiment score as routing accuracy or fluency as factual correctness.

**Your answer:** I choose routing short English customer support messages to three departments. This is a text classification task, so an encoder model like DistilBERT fits best. The sentiment checkpoint does not fit, because it only knows POSITIVE and NEGATIVE, not departments, so it must be fine-tuned with three department labels. One test case: “I was charged twice for my order” should go to Billing. I would check accuracy and F1 per department on real labeled messages, not the sentiment score. If the model is not confident or a message has two problems, it should go to a human.

## Optional investigation - choose one (10–15 minutes)

These preserve the original exploration topics. A model-card review is enough for the first pass; download only after checking size and language support. If you run a model, use an explicit model ID and record its commit revision, task, tokenizer, library version and actual output. If you do not run it, state **not executed** and explain the planned test.

| Option | Action | Evidence to submit |
|---|---|---|
| Another language | Find a text-generation model whose card supports a language you know. | Model URL, language, architecture, task, weight size, limitation and one bounded test prompt. |
| Cased BERT | Inspect [google-bert/bert-base-cased](https://huggingface.co/google-bert/bert-base-cased). Identify its mask marker and contrast cased/uncased tokenization. | The marker and source; if executed, the top three observed candidates for one input. Do not copy a prediction from a widget. |
| POS tagging | Inspect [vblagoje/bert-english-uncased-finetuned-pos](https://huggingface.co/vblagoje/bert-english-uncased-finetuned-pos) or another English POS checkpoint. | Task, at least three tag meanings, and why those labels differ from NER. Note any missing license/training information in the card. |
| Translation | Choose one directed language pair and inspect a suitable model card. | Source/target language, tokenizer requirements, model URL and a plan to check numbers, names and meaning. |
| Original advanced demos | Enable one optional section in the guided lab: zero-shot, NER, QA or summary. | Your observed result and one limitation demonstrated by changing the input. |

**Your investigation and execution status:** TODO

In [15]:
urdu_gen = pipeline('text-generation', model='Imran1/gpt2-urdu-news', device=-1)
print(urdu_gen.model.config._commit_hash, type(urdu_gen.model).__name__, type(urdu_gen.tokenizer).__name__)
set_seed(SEED)
print(urdu_gen('پاکستان میں آج', max_new_tokens=15, do_sample=True, pad_token_id=urdu_gen.tokenizer.eos_token_id))

config.json:   0%|          | 0.00/909 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/663M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/730 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/470 [00:00<?, ?B/s]

Device set to use cpu


9020620e702201f2676b172b505a940b2eddd63e GPT2LMHeadModel GPT2TokenizerFast
[{'generated_text': 'پاکستان میں آج کے بعد ہی ارب روپے کا اضافہ ہوا ہے جو'}]


**Model:** https://huggingface.co/Imran1/gpt2-urdu-news, revision 9020620e702201f2676b172b505a940b2eddd63e.

**Language:** Urdu.
**Architecture:** GPT-2, decoder only (GPT2LMHeadModel with GPT2TokenizerFast).

**Task:** text generation. Weight size: about 663 MB for the model.safetensors file. Test prompt: "پاکستان میں آج", which means “In Pakistan today”, with max 15 new tokens. Output: "پاکستان میں آج کے بعد ہی ارب روپے کا اضافہ ہوا ہے جو", which basically means “in Pakistan today after that billion rupees increase has happened which”. The text sounds like news, but it stops in the middle and does not fully make sense. Limitation: the model card says it was fine-tuned on only 1000 samples, so the Urdu output is actually weak.

In [17]:
cased = pipeline('fill-mask', model='google-bert/bert-base-cased', device=-1)   # cased BERT model
print(cased.model.config._commit_hash, type(cased.model).__name__, type(cased.tokenizer).__name__)   # revision, model, tokenizer
print('Mask marker:', cased.tokenizer.mask_token)

# same sentence with the cased and the uncased tokenizer
text = "sonya lives in Latvia and city is Riga"
print('Cased:', cased.tokenizer.tokenize(text))
print('Uncased:', unmasked.tokenizer.tokenize(text))

print(cased(f"Paris is the capital of {cased.tokenizer.mask_token}.", top_k=3))   # top three candidates

Some weights of the model checkpoint at google-bert/bert-base-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


cd5ef92a9fb2f889e972770a36d4ed042daf221e BertForMaskedLM BertTokenizerFast
Mask marker: [MASK]
Cased: ['son', '##ya', 'lives', 'in', 'Latvia', 'and', 'city', 'is', 'Riga']
Uncased: ['sonya', 'lives', 'in', 'latvia', 'and', 'city', 'is', 'riga']
[{'score': 0.9695423245429993, 'token': 1699, 'token_str': 'France', 'sequence': 'Paris is the capital of France.'}, {'score': 0.01791955530643463, 'token': 1980, 'token_str': 'Europe', 'sequence': 'Paris is the capital of Europe.'}, {'score': 0.0018615867011249065, 'token': 2123, 'token_str': 'Paris', 'sequence': 'Paris is the capital of Paris.'}]


I used google-bert/bert-base-cased with revision cd5ef92a9fb2f889e972770a36d4ed042daf221e, which is a BERT encoder model, and its mask marker is [MASK].
I tokenized “sonya lives in Latvia and city is Riga” with both tokenizers. The cased tokenizer kept the capital letters in “Latvia” and “Riga”, but the uncased tokenizer basically changed everything to lowercase, like “latvia” and “riga”. Also the cased tokenizer split “sonya” into two tokens, “son” and “##ya”, while the uncased tokenizer actually kept “sonya” as one token, so the same word can be split differently by different tokenizers. For “Paris is the capital of [MASK].” the top three guesses were France with a score of 0.970, Europe with 0.018 and Paris with 0.002, so the model was very sure about France. The warning about unused weights is normal, because those parts are not needed for fill-mask.

## Submission check

- Replace every required `TODO` with your own code or interpretation.
- Restart and run the notebook in order using the pinned kernel. Keep the printed versions.
- Keep your observed outputs; do not invent a plausible result for an unexecuted cell.
- Explain one failure or ambiguity even when the code runs successfully.
- Mark the optional investigation as executed or not executed.

**Assessment:** task/model mapping (2 points), token and mask evidence (2), sentiment interpretation (2), bounded generation and decoding comparison (3), model decision and limitation (1). Total: **10 points**. Optional work is for discussion, not required to achieve 10/10.

## Sources and attribution

The original notebook follows the Hugging Face course chapter [Transformers, what can they do?](https://huggingface.co/learn/llm-course/chapter1/3). This revision uses new classroom examples and adds setup, model selection, bounded generation, exercises and interpretation. This is **source attribution**, not a license grant; the course source and each model retain their own applicable licenses. Model weights are downloaded separately and are not distributed in this notebook.

Primary references checked for this revision:

- [Transformers 4.57.1 pipeline API](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines), including tasks, NER aggregation and output formats.
- [Transformers 4.57.1 generation parameters](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/text_generation), including sampling and token limits.
- [Installation, caching and offline mode](https://huggingface.co/docs/transformers/v4.57.1/en/installation).
- [Sentiment model card](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english) · [masked-language model card](https://huggingface.co/distilbert/distilbert-base-uncased) · [DistilGPT2 model card](https://huggingface.co/distilbert/distilgpt2).
- Weight-size evidence: [sentiment files](https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english/tree/main), [mask-model files](https://huggingface.co/distilbert/distilbert-base-uncased/tree/main), [generation-model files](https://huggingface.co/distilbert/distilgpt2/tree/main). The figures are for one safetensors weight file, not the entire repository or runtime memory.

Model revision hashes in the code freeze the reviewed files. A fixed seed improves repeatability in a fixed environment; it does not guarantee bit-identical text across different hardware, library versions or decoding implementations. Inspect warnings; this notebook does not suppress them globally.